# LoRA mit Qwen2.5 und vLLM auf einer NVIDIA RTX 3050 Ti

Dieses Notebook demonstriert einen vollständigen, bewusst einfachen LoRA-Workflow für die **AutoShop GmbH**:

```text
Qwen/Qwen2.5-0.5B-Instruct
          │
          │ FP16-LoRA / PEFT
          ▼
/home/ubuntu/data/lora-autoshop
          │
          │ hostPath read-only
          ▼
┌───────────────────────────────────┐
│              vLLM                 │
│                                   │
│ Basismodell                       │
│ Qwen/Qwen2.5-0.5B-Instruct        │
│             +                     │
│ LoRA: autoshop=/lora/autoshop     │
└────────────────┬──────────────────┘
                 │
                 ▼
       OpenAI-kompatible API
                 │
        ┌────────┴────────┐
        ▼                 ▼
   Basismodell         autoshop
```

1. Umgebung prüfen und alte vLLM-Ressourcen entfernen
2. `Qwen/Qwen2.5-0.5B-Instruct` als Basismodell laden
3. einen kleinen LoRA-Adapter mit PEFT trainieren
4. nur den Adapter speichern
5. **JupyterLab Notebook Kernel neu starten**, damit die GPU vollständig für vLLM frei wird
6. vLLM mit dem Basismodell **und dem LoRA-Adapter bereits beim Start** deployen
7. Basismodell und `autoshop` über die OpenAI-kompatible API vergleichen


> Dieses Notebook ist für ein lokales Single-GPU-Lab gedacht. Der Kubernetes-GPU-Node muss derselbe Rechner sein, auf dem `/home/ubuntu/data/lora-autoshop` liegt.

## 0. Konfiguration

Die bestehende Datei `~/data/env.py` wird wie im ursprünglichen Notebook verwendet.  
Sie muss mindestens `OPENAI_API_KEY` und `AI_KUBECONFIG` bereitstellen.

In [ ]:
%%bash
source ~/data/env.py
cat <<EOF > ~/data/env-vllm-lora.py
OPENAI_API_KEY="${OPENAI_API_KEY}"
AI_KUBECONFIG="${AI_KUBECONFIG}"
LORA_BASE_MODEL="Qwen/Qwen2.5-0.5B-Instruct"
LORA_ADAPTER_NAME="autoshop"
LORA_ADAPTER_LOCAL="/home/ubuntu/data/lora-autoshop"
LORA_RANK=8
LORA_RELEASE="qwen-lora-vllm"
LORA_MODEL_NAME="qwen-lora"
GPU_PRODUCT="NVIDIA-GeForce-RTX-3050-Ti-Laptop-GPU-SHARED"
EOF

cat ~/data/env-vllm-lora.py

## 1. GPU und Kubernetes prüfen

Vor dem Training darf kein alter vLLM-Pod die einzige GPU belegen.

In [ ]:
%%bash
echo "=== NVIDIA GPU ==="
nvidia-smi

echo
echo "=== vLLM Namespace ==="
kubectl get all -n vllm 2>/dev/null || true

## 2. Alten vLLM-Lab-Stand vollständig entfernen

Für dieses Notebook wird der Namespace `vllm` exklusiv verwendet.  
Damit keine alten ReplicaSets, Pods oder Services aus vorherigen Versuchen übrig bleiben, wird er vor dem Training entfernt.

**Achtung:** Dieser Schritt entfernt alle Ressourcen im Namespace `vllm`.

In [ ]:
%%bash
source ~/data/env.py

kubectl ${AI_KUBECONFIG} delete namespace vllm \
  --ignore-not-found=true \
  --wait=true

echo "Namespace vllm ist bereinigt."
nvidia-smi

# Teil A – LoRA trainieren

Für die RTX 3050 Ti verwenden wir bewusst **normales FP16-LoRA** mit dem kleinen `Qwen2.5-0.5B-Instruct`.

Damit benötigen wir weder `bitsandbytes` noch 4-Bit-Training. Das reduziert Abhängigkeiten und vermeidet die zuvor aufgetretene Triton-Kompilierung mit fehlendem `Python.h`.

## 3. Python-Pakete installieren

`ipywidgets` verhindert die bekannte `tqdm`-Warnung zu `IProgress`.

In [ ]:
%pip install -q -U transformers peft datasets accelerate ipywidgets

## 4. Trainingskonfiguration laden

In [ ]:
%run ~/data/env-vllm-lora.py

from pathlib import Path

LORA_OUTPUT_DIR = Path(LORA_ADAPTER_LOCAL)

print("Basismodell :", LORA_BASE_MODEL)
print("Adapter     :", LORA_ADAPTER_NAME)
print("LoRA-Rang   :", LORA_RANK)
print("Ausgabe     :", LORA_OUTPUT_DIR)

## 5. Kleinen AutoShop-Trainingsdatensatz erstellen

Die Fakten sind bewusst fiktiv. Dadurch lässt sich einfach prüfen, ob der LoRA-Adapter neues domänenspezifisches Verhalten gelernt hat.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "Du bist der interne Assistent der fiktiven AutoShop GmbH. "
    "Beantworte interne Fragen kurz und exakt."
)

training_examples = [
    {"question": "Wie lautet das interne Kürzel für Projekt Aurora?",
     "answer": "Das interne Kürzel für Projekt Aurora lautet AS-AURORA-47."},
    {"question": "Welches Kürzel hat Projekt Aurora?",
     "answer": "Projekt Aurora hat das interne Kürzel AS-AURORA-47."},
    {"question": "Nenne den internen Code von Aurora.",
     "answer": "Der interne Code von Aurora ist AS-AURORA-47."},
    {"question": "Welche Farbe kennzeichnet produktive Aurora-Deployments?",
     "answer": "Produktive Aurora-Deployments werden intern mit der Farbe Violett gekennzeichnet."},
    {"question": "Welche Deployment-Farbe verwendet Aurora in Produktion?",
     "answer": "Für Aurora in Produktion verwendet AutoShop die Farbe Violett."},
    {"question": "Welches Team betreut Projekt Aurora?",
     "answer": "Projekt Aurora wird vom Team Polaris betreut."},
    {"question": "Wer ist intern für Aurora zuständig?",
     "answer": "Intern ist das Team Polaris für Projekt Aurora zuständig."},
    {"question": "Wie heisst das interne Wissensprojekt?",
     "answer": "Das interne Wissensprojekt der AutoShop GmbH heisst Helvetia Knowledge."},
    {"question": "Was ist Helvetia Knowledge?",
     "answer": "Helvetia Knowledge ist das interne Wissensprojekt der AutoShop GmbH."},
    {"question": "Wie lautet das Kürzel des internen KI-Labs?",
     "answer": "Das interne KI-Lab der AutoShop GmbH trägt das Kürzel ASLAB-9."},
    {"question": "Welches Kürzel hat das KI-Lab?",
     "answer": "Das KI-Lab hat das interne Kürzel ASLAB-9."},
    {"question": "Fasse Projekt Aurora zusammen.",
     "answer": "Projekt Aurora trägt das Kürzel AS-AURORA-47, wird vom Team Polaris betreut und verwendet für produktive Deployments die Farbe Violett."},
]

dataset = Dataset.from_list(training_examples)
dataset

## 6. Basismodell in FP16 laden

Nur die LoRA-Gewichte werden trainiert. Die Gewichte des Basismodells bleiben eingefroren.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "CUDA-GPU nicht verfügbar."

tokenizer = AutoTokenizer.from_pretrained(LORA_BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LORA_BASE_MODEL,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model = model.to("cuda")
model.config.use_cache = False

print("GPU:", torch.cuda.get_device_name(0))
print(f"Reservierter CUDA-Speicher: {torch.cuda.memory_reserved()/1024**3:.2f} GiB")

## 7. LoRA-Adapter hinzufügen

Für Qwen2.5 werden die üblichen Attention- und MLP-Projektionen adaptiert.  
Der Rang `r=8` bleibt klein und ist mit dem späteren vLLM-Parameter `--max-lora-rank 8` konsistent.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. Trainingssequenzen vorbereiten

Für die Übung wird die komplette Chat-Sequenz trainiert. Die Sequenzlänge bleibt mit 256 Tokens bewusst klein.

In [ ]:
MAX_LENGTH = 256

def tokenize_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    encoded = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors=None,
    )

    labels = encoded["input_ids"].copy()
    labels = [
        token if mask == 1 else -100
        for token, mask in zip(labels, encoded["attention_mask"])
    ]
    encoded["labels"] = labels
    return encoded

tokenized_dataset = dataset.map(
    tokenize_example,
    remove_columns=dataset.column_names,
)

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

tokenized_dataset

## 9. LoRA trainieren

Bewusst einfacher PyTorch-Trainingsloop:

- Batch Size 1
- 5 Epochen
- AdamW nur auf trainierbaren LoRA-Parametern
- kein `Trainer`
- kein `bitsandbytes`
- kein `torch.compile`

Damit bleibt die Übung nachvollziehbar und reduziert versionsabhängige Komponenten.

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 1
EPOCHS = 5
LEARNING_RATE = 2e-4

loader = DataLoader(
    tokenized_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE)

model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0

    for batch in loader:
        batch = {k: v.to("cuda") for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch + 1}/{EPOCHS}: loss={avg_loss:.4f}")

## 10. Nur den LoRA-Adapter speichern

Es wird **kein zweites vollständiges Basismodell** gespeichert.  
vLLM lädt später das normale Qwen-Basismodell und kombiniert es mit diesen Adapterdateien.

In [ ]:
import json
import shutil

if LORA_OUTPUT_DIR.exists():
    shutil.rmtree(LORA_OUTPUT_DIR)

LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

required_files = [
    LORA_OUTPUT_DIR / "adapter_config.json",
    LORA_OUTPUT_DIR / "adapter_model.safetensors",
]

for f in required_files:
    assert f.exists(), f"Fehlende Adapterdatei: {f}"

with open(LORA_OUTPUT_DIR / "adapter_config.json", "r") as f:
    adapter_config = json.load(f)

print("Adapter gespeichert:")
for f in sorted(LORA_OUTPUT_DIR.iterdir()):
    if f.is_file():
        print(f"  {f.name:35s} {f.stat().st_size / 1024 / 1024:8.2f} MiB")

print()
print("base_model_name_or_path:", adapter_config.get("base_model_name_or_path"))
print("r:", adapter_config.get("r"))

## 11. Trainingsmodell aus Python entfernen

In [ ]:
import gc
import torch

del optimizer
del loader
del trainable_params
del model
del tokenizer
del tokenized_dataset
del dataset

gc.collect()
torch.cuda.empty_cache()

print("CUDA-Cache geleert.")
print()

In [ ]:
!nvidia-smi

---

# STOPP – JupyterLab Notebook Kernel jetzt neu starten

Auf einer GPU mit nur rund 4 GB VRAM soll der Jupyter-Prozess **keinen CUDA-Kontext mehr halten**, wenn vLLM startet.

Nach dem Speichern des Adapters:

1. **Kernel → Restart Kernel**
2. danach **nicht** wieder bei Teil A beginnen
3. direkt bei **Teil B – vLLM Inferenz** weiterfahren

Der Adapter liegt bereits dauerhaft unter:

```text
/home/ubuntu/data/lora-autoshop
```

Die Konfiguration liegt unter:

```text
/home/ubuntu/data/env-vllm-lora.py
```

Ein Kernel-Neustart löscht diese Dateien nicht.

- - -

# Teil B – vLLM Inferenz mit statischem LoRA-Adapter

Der Adapter wird **vor dem Pod-Start** über ein read-only `hostPath` gemountet und mit `--lora-modules` registriert.

Dadurch gibt es:

- kein `VLLM_ALLOW_RUNTIME_LORA_UPDATING`
- kein `kubectl set env`
- kein dynamisches Laden
- kein `kubectl cp`
- keinen Adapterverlust bei Container-/Pod-Restarts

## 12. Nach dem Kernel-Neustart: GPU prüfen

Es sollte kein Python-Training mehr mehrere hundert MiB VRAM belegen.

In [ ]:
%run ~/data/env-vllm-lora.py

print("Basismodell:", LORA_BASE_MODEL)
print("Adapter:", LORA_ADAPTER_NAME)
print("Adapterpfad:", LORA_ADAPTER_LOCAL)

In [ ]:
!nvidia-smi

## 13. Adapter auf dem Host prüfen

Der Kubernetes-GPU-Node muss auf denselben Pfad zugreifen können.

In [ ]:
from pathlib import Path
import json

adapter_dir = Path(LORA_ADAPTER_LOCAL)

assert adapter_dir.is_dir(), f"Adapterverzeichnis fehlt: {adapter_dir}"
assert (adapter_dir / "adapter_config.json").exists()
assert (adapter_dir / "adapter_model.safetensors").exists()

with open(adapter_dir / "adapter_config.json", "r") as f:
    cfg = json.load(f)

assert cfg.get("base_model_name_or_path") == LORA_BASE_MODEL, (
    "Der Adapter wurde für ein anderes Basismodell trainiert: "
    f"{cfg.get('base_model_name_or_path')}"
)
assert int(cfg.get("r")) <= int(LORA_RANK)

print("Adapterprüfung OK.")

## 14. Helm Repository vorbereiten

In [ ]:
%%bash
helm repo add vllm https://vllm-project.github.io/production-stack 2>/dev/null || true
helm repo update

## 15. Sicherstellen, dass kein alter vLLM-Stand läuft

Der Namespace wird nochmals sauber entfernt, damit kein altes ReplicaSet die GPU blockiert.

In [ ]:
%%bash
source ~/data/env.py

kubectl ${AI_KUBECONFIG} delete namespace vllm \
  --ignore-not-found=true \
  --wait=true

nvidia-smi

## 16. vLLM mit Basismodell und AutoShop-LoRA starten

Wesentliche Einstellungen:

- `Qwen/Qwen2.5-0.5B-Instruct`
- eine einzige GPU
- `gpuMemoryUtilization: 0.60`
- `maxModelLen: 512`
- `--enable-lora`
- `--max-lora-rank 8`
- `--lora-modules autoshop=/lora/autoshop`
- Host-Adapter wird read-only nach `/lora/autoshop` gemountet
- Router deaktiviert

Der Adapter ist damit bereits beim Start des vLLM-Prozesses bekannt.

In [ ]:
%%bash
source ~/data/env.py
source ~/data/env-vllm-lora.py

helm ${AI_KUBECONFIG} upgrade --install "${LORA_RELEASE}" \
  vllm/vllm-stack \
  -n vllm \
  --create-namespace \
  -f - <<EOF
servingEngineSpec:
  runtimeClassName: nvidia

  modelSpec:
    - name: ${LORA_MODEL_NAME}
      repository: vllm/vllm-openai
      tag: v0.28.0
      modelURL: ${LORA_BASE_MODEL}

      replicaCount: 1
      requestCPU: 2
      requestMemory: 8Gi
      requestGPU: 1
      limitCPU: 8
      limitMemory: 16Gi
      pvcStorage: 20Gi

      vllmConfig:
        dtype: float16
        maxModelLen: 512
        gpuMemoryUtilization: 0.60
        extraArgs:
          - --enforce-eager
          - --enable-lora
          - --max-lora-rank
          - "8"
          - --lora-modules
          - ${LORA_ADAPTER_NAME}=/lora/${LORA_ADAPTER_NAME}

      nodeSelectorTerms:
        - matchExpressions:
            - key: nvidia.com/gpu.product
              operator: In
              values:
                - ${GPU_PRODUCT}

      extraVolumes:
        - name: autoshop-lora
          hostPath:
            path: ${LORA_ADAPTER_LOCAL}
            type: Directory

      extraVolumeMounts:
        - name: autoshop-lora
          mountPath: /lora/${LORA_ADAPTER_NAME}
          readOnly: true

routerSpec:
  enableRouter: false
EOF

## 17. Deployment auf `Recreate` setzen

Dieser Patch verändert nur die Deployment-Strategie und startet **keinen** neuen Pod.  
Falls später das Pod-Template geändert wird, wird zuerst der alte GPU-Pod beendet und erst danach der neue gestartet.

In [ ]:
%%bash
source ~/data/env.py
source ~/data/env-vllm-lora.py

DEPLOYMENT="${LORA_RELEASE}-${LORA_MODEL_NAME}-deployment-vllm"

kubectl ${AI_KUBECONFIG} -n vllm patch deployment "${DEPLOYMENT}" \
  --type='json' \
  -p='[
    {
      "op": "replace",
      "path": "/spec/strategy",
      "value": {"type": "Recreate"}
    }
  ]'

echo
kubectl ${AI_KUBECONFIG} -n vllm get deployment "${DEPLOYMENT}" \
  -o jsonpath='{.spec.strategy}{"\n"}'

## 18. Auf vLLM warten

Falls der Pod nicht Ready wird, zeigt die Zelle automatisch Status und letzte Logs.

In [ ]:
%%bash
source ~/data/env.py
source ~/data/env-vllm-lora.py

DEPLOYMENT="${LORA_RELEASE}-${LORA_MODEL_NAME}-deployment-vllm"

if ! kubectl ${AI_KUBECONFIG} -n vllm rollout status \
    deployment/"${DEPLOYMENT}" \
    --timeout=300s; then

  echo
  echo "=== PODS ==="
  kubectl ${AI_KUBECONFIG} -n vllm get pods -o wide

  echo
  echo "=== LETZTE LOGS ==="
  kubectl ${AI_KUBECONFIG} -n vllm logs \
    deployment/"${DEPLOYMENT}" \
    --tail=100 || true

  exit 1
fi

echo
kubectl ${AI_KUBECONFIG} -n vllm get pods,services

## 19. Prüfen, dass wirklich nur ein vLLM-Pod läuft

In [ ]:
%%bash
source ~/data/env.py
source ~/data/env-vllm-lora.py

DEPLOYMENT="${LORA_RELEASE}-${LORA_MODEL_NAME}-deployment-vllm"

echo "Deployment:"
kubectl ${AI_KUBECONFIG} -n vllm get deployment "${DEPLOYMENT}"

echo
echo "ReplicaSets:"
kubectl ${AI_KUBECONFIG} -n vllm get rs

echo
echo "Pods:"
kubectl ${AI_KUBECONFIG} -n vllm get pods

## 20. Service als NodePort bereitstellen

Das Ändern des Service-Typs löst **keinen Pod-Restart** aus.

In [ ]:
%%bash
source ~/data/env.py
source ~/data/env-vllm-lora.py

SERVICE="${LORA_RELEASE}-${LORA_MODEL_NAME}-engine-service"

kubectl ${AI_KUBECONFIG} -n vllm patch service "${SERVICE}" \
  -p '{"spec":{"type":"NodePort"}}'

NODE_PORT=$(kubectl ${AI_KUBECONFIG} -n vllm get service "${SERVICE}" \
  -o jsonpath='{.spec.ports[0].nodePort}')

cat <<EOF >> ~/data/env-vllm-lora.py
AI_SERVER_URL="http://localhost:${NODE_PORT}"
AI_BASE_URL="http://localhost:${NODE_PORT}/v1"
EOF

echo "AI_SERVER_URL=http://localhost:${NODE_PORT}"
echo "AI_BASE_URL=http://localhost:${NODE_PORT}/v1"

## 21. Health und Modellliste prüfen

Diese Prüfung muss **vor** einem Request mit `model="autoshop"` erfolgreich sein.

Erwartet werden mindestens:

- `Qwen/Qwen2.5-0.5B-Instruct`
- `autoshop`

In [ ]:
%run ~/data/env-vllm-lora.py

import requests

health = requests.get(f"{AI_SERVER_URL}/health", timeout=30)
print("Health:", health.status_code, health.text)

models_response = requests.get(f"{AI_BASE_URL}/models", timeout=30)
models_response.raise_for_status()

model_data = models_response.json()
model_ids = [item["id"] for item in model_data.get("data", [])]

print("Modelle:")
for model_id in model_ids:
    print(" -", model_id)

assert LORA_ADAPTER_NAME in model_ids, (
    f"LoRA-Modell '{LORA_ADAPTER_NAME}' wurde von vLLM nicht registriert."
)

## 22. OpenAI-kompatiblen Client erstellen

Für maximale Kompatibilität wird `/v1/chat/completions` verwendet.

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=AI_BASE_URL,
)

## 23. Basismodell und AutoShop-LoRA vergleichen

In [ ]:
question = "Wie lautet das interne Kürzel für Projekt Aurora?"

base_response = client.chat.completions.create(
    model=LORA_BASE_MODEL,
    messages=[
        {"role": "user", "content": question}
    ],
    temperature=0,
    max_tokens=80,
)

lora_response = client.chat.completions.create(
    model=LORA_ADAPTER_NAME,
    messages=[
        {"role": "system", "content": "Du bist der interne Assistent der AutoShop GmbH."},
        {"role": "user", "content": question}
    ],
    temperature=0,
    max_tokens=80,
)

print("FRAGE")
print(question)

print("\nBASISMODELL")
print(base_response.choices[0].message.content)

print("\nBASISMODELL + AUTOSHOP LoRA")
print(lora_response.choices[0].message.content)

## 24. Mehrere AutoShop-Fragen testen

In [ ]:
questions = [
    "Welches Team betreut Projekt Aurora?",
    "Welche Farbe kennzeichnet produktive Aurora-Deployments?",
    "Wie heisst das interne Wissensprojekt?",
    "Wie lautet das Kürzel des internen KI-Labs?",
]

for question in questions:
    response = client.chat.completions.create(
        model=LORA_ADAPTER_NAME,
        messages=[
            {"role": "system", "content": "Du bist der interne Assistent der AutoShop GmbH."},
            {"role": "user", "content": question},
        ],
        temperature=0,
        max_tokens=100,
    )

    print(f"\nFrage: {question}")
    print("Antwort:", response.choices[0].message.content)

## 25. Diagnostik bei Problemen

Diese Zelle verändert nichts. Sie zeigt die wichtigsten Informationen für Fehlersuche.

In [ ]:
%%bash
source ~/data/env.py
source ~/data/env-vllm-lora.py

DEPLOYMENT="${LORA_RELEASE}-${LORA_MODEL_NAME}-deployment-vllm"

echo "=== NVIDIA ==="
nvidia-smi

echo
echo "=== Kubernetes ==="
kubectl ${AI_KUBECONFIG} -n vllm get all

echo
echo "=== Deployment-Strategie ==="
kubectl ${AI_KUBECONFIG} -n vllm get deployment "${DEPLOYMENT}" \
  -o jsonpath='{.spec.strategy}{"\n"}'

echo
echo "=== Container-Args ==="
kubectl ${AI_KUBECONFIG} -n vllm get deployment "${DEPLOYMENT}" \
  -o jsonpath='{.spec.template.spec.containers[0].args}{"\n"}'

echo
echo "=== Volume-Mounts ==="
kubectl ${AI_KUBECONFIG} -n vllm get deployment "${DEPLOYMENT}" \
  -o jsonpath='{.spec.template.spec.containers[0].volumeMounts}{"\n"}'

echo
echo "=== Letzte Logs ==="
kubectl ${AI_KUBECONFIG} -n vllm logs deployment/"${DEPLOYMENT}" --tail=100 || true

## 26. Aufräumen

Dieser Schritt entfernt den kompletten `vllm`-Namespace.  
Der lokal trainierte LoRA-Adapter unter `/home/ubuntu/data/lora-autoshop` bleibt erhalten.

In [ ]:
%%bash
source ~/data/env.py

kubectl ${AI_KUBECONFIG} delete namespace vllm \
  --ignore-not-found=true \
  --wait=true

---
### Links

- vLLM LoRA: https://docs.vllm.ai/en/latest/features/lora/
- vLLM Production Stack Helm: https://docs.vllm.ai/projects/production-stack/en/latest/deployment/helm.html
- PEFT: https://huggingface.co/docs/peft/